In [1]:
from pathlib import Path
import anndata as ad
import pandas as pd
import numpy as np
import scipy.sparse as sp
import networkx as nx
import scglue
import scglue.data as gld
import scglue.genomics as gl
import json

# ============================
# Paths
# ============================
rna_path  = Path("./data/rna_subsampled.h5ad")
atac_path = Path("./data/atac_subsampled.h5ad")
gtf_path  = Path("./data/gencode.v48lift37.annotation.gtf.gz")

out_dir   = Path("./data")
out_dir.mkdir(exist_ok=True)
graph_gml = out_dir / "guidance.graphml.gz"
graph_pkl = out_dir / "guidance.gpickle"

print(">>> Loading subsampled RNA & ATAC ...")
rna  = ad.read_h5ad(rna_path)
atac = ad.read_h5ad(atac_path)

# ---- ensure CSR & float32 ----
if not sp.isspmatrix_csr(rna.X):  rna.X  = sp.csr_matrix(rna.X)
if not sp.isspmatrix_csr(atac.X): atac.X = sp.csr_matrix(atac.X)
rna.X  = rna.X.astype(np.float32)
atac.X = atac.X.astype(np.float32)

# ---- Backup counts layer if missing ----
if "counts" not in rna.layers:
    rna.layers["counts"] = rna.X.copy()
if "counts" not in atac.layers:
    atac.layers["counts"] = atac.X.copy()

# ===============================
# 1) Annotate RNA gene coordinates
# ===============================
print(">>> Annotating RNA genes using GTF")
gld.get_gene_annotation(rna, gtf=str(gtf_path), gtf_by="gene_name")

rna = rna[:, rna.var[["chrom","chromStart","chromEnd"]].notna().all(axis=1)].copy()
print("RNA annotated genes:", rna.n_vars)

# ===============================
# 2) Parse ATAC peak coordinates
# ===============================
print(">>> Parsing ATAC peak coordinates ...")
parts = atac.var_names.to_series().str.split(r"[:-]", n=2, expand=True)

if parts.shape[1] != 3:
    raise ValueError("ATAC peak format must be chr-start-end")

atac.var["chrom"]      = parts[0].values
atac.var["chromStart"] = parts[1].astype(int).values
atac.var["chromEnd"]   = parts[2].astype(int).values
print("ATAC peaks:", atac.n_vars)

# ===============================
# 3) Build guidance graph
# ===============================
print(">>> Building RNA-anchored guidance graph ...")
G = gl.rna_anchored_guidance_graph(rna, atac)

print("Graph nodes:", G.number_of_nodes(), "edges:", G.number_of_edges())

# ===============================
# 4) Sanitize (make H5AD-safe)
# ===============================
def make_h5ad_safe(adata):
    adata.obs_names = adata.obs_names.astype(str)
    adata.var_names = adata.var_names.astype(str)
    adata.var_names_make_unique()

    def fix_df(df):
        df = df.copy()
        obj_cols = df.select_dtypes(include=["object"]).columns
        
        # Try numeric conversion
        for c in obj_cols:
            df[c] = pd.to_numeric(df[c], errors="ignore")

        # Fix string columns & objects
        for c in df.columns:
            if pd.api.types.is_string_dtype(df[c].dtype) and not pd.api.types.is_object_dtype(df[c].dtype):
                df[c] = df[c].astype(object)
                df[c] = df[c].where(df[c].notna(), "")
            
            if pd.api.types.is_object_dtype(df[c]):
                df[c] = df[c].apply(
                    lambda x: json.dumps(x) if isinstance(x, (list, tuple, dict, set, np.ndarray))
                    else ("" if x is None else str(x))
                )
        return df

    adata.obs = fix_df(adata.obs)
    adata.var = fix_df(adata.var)

print(">>> Sanitizing metadata ...")
make_h5ad_safe(rna)
make_h5ad_safe(atac)

# ===============================
# 5) Save outputs
# ===============================
print(">>> Saving guidance graph ...")
nx.write_graphml(G, graph_gml)
nx.write_gpickle(G, graph_pkl)

print("All done ✓")
print("Graph saved to:", graph_gml)


/home/mza/conda_envs/glue/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/mza/conda_envs/glue/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/home/mza/conda_envs/glue/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/home/mza/conda_envs/glue/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_hdf from `anndata` is deprecated. Import anndata.io.read_hdf instead.
  warnings.warn(msg, FutureWarning)
/home/mza/conda_envs/glue

>>> Loading subsampled RNA & ATAC ...
>>> Annotating RNA genes using GTF
RNA annotated genes: 293
>>> Parsing ATAC peak coordinates ...
ATAC peaks: 543960
>>> Building RNA-anchored guidance graph ...


window_graph: 100%|██████████| 293/293 [00:09<00:00, 30.11it/s]


Graph nodes: 544253 edges: 569363
>>> Sanitizing metadata ...
>>> Saving guidance graph ...
All done ✓
Graph saved to: data/guidance.graphml.gz
